# Danish Electricity Prices ⚡🇩🇰
Forecast tomorrow's **hourly spot price in DK1** (Jutland and Funen, Aalborg's price area) with real data from [Energi Data Service](https://www.energidataservice.dk/), Energinet's free open-data API (no key needed).

You'll find out what every energy trader knows: **price history alone barely beats "same as yesterday". The weather forecast is the feature.**

Guide note: [[Danish Electricity Prices]] · background: [[Time Series Forecasting]], [[Time Series Validation and Features]], [[Ensemble Methods]]

**Setup:** hourly prices for **Oct 2023 – Sep 2025** (the frozen `Elspotprices` dataset, so your numbers match these). We train on Oct 2023 – Dec 2024 and test on **all of 2025**. At the end there's an optional **live** cell that forecasts the next day from today's data.

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import json, pathlib, urllib.request, urllib.parse, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

PRICE_AREA = "DK1"        # DK1 = west of the Great Belt (Aalborg), DK2 = east (Copenhagen)
CACHE = pathlib.Path("data"); CACHE.mkdir(exist_ok=True)

def energinet(dataset, start, end, area=PRICE_AREA):
    """Download a dataset from api.energidataservice.dk (cached as CSV)."""
    f = CACHE / f"{dataset}_{area}_{start}_{end}.csv"
    if not f.exists():
        q = {"start": start, "end": end, "filter": json.dumps({"PriceArea": [area]}), "limit": 0}
        url = f"https://api.energidataservice.dk/dataset/{dataset}?" + urllib.parse.urlencode(q)
        pd.DataFrame(json.load(urllib.request.urlopen(url))["records"]).to_csv(f, index=False)
    return pd.read_csv(f)

prices = energinet("Elspotprices", "2023-10-01", "2025-10-01")
fc = energinet("Forecasts_Hour", "2023-10-01", "2025-10-01")
fc["t"] = pd.to_datetime(fc.HourUTC)
y = prices.assign(t=pd.to_datetime(prices.HourUTC)).set_index("t").SpotPriceEUR.sort_index().asfreq("h")
df = pd.DataFrame({"y": y})                                     # index = hour in UTC, y = EUR/MWh
print(f"{len(df):,} hours, {df.y.isna().sum()} missing, {(df.y < 0).sum()} hours with NEGATIVE prices, max {df.y.max():.0f} EUR/MWh")
fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
df.y.resample("D").mean().plot(ax=ax[0], title="daily mean price (EUR/MWh)")
local_hour = df.index.tz_localize("UTC").tz_convert("Europe/Copenhagen").hour
df.y.groupby(local_hour).mean().plot(ax=ax[1], marker="o", title="average by hour of day (Danish time)"); plt.show()

## 1. Calendar features in *Danish* time
People cook at 17:00 Danish time, not UTC, and daylight saving time moves UTC by an hour twice a year. Convert the UTC index to `Europe/Copenhagen` and add `hour`, `dow` (Monday = 0), `month` and `weekend` (1 on Saturday/Sunday).

In [ ]:
def add_calendar(df):
    # TODO 1: local = index converted UTC → Europe/Copenhagen; add hour, dow, month, weekend columns
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_cal():
    idx = pd.to_datetime(["2024-03-31 00:00", "2024-03-31 01:00", "2024-06-01 10:00"])   # DST starts 31 Mar at 01:00 UTC
    out = add_calendar(pd.DataFrame({"y": [1, 2, 3]}, index=idx))
    return out.hour.tolist() == [1, 3, 12] and out.dow.tolist() == [6, 6, 5] and out.weekend.tolist() == [1, 1, 1] and out.month.tolist() == [3, 3, 6]
check("calendar", _t_cal, "00:00 UTC = 01:00 CET, 01:00 UTC = 03:00 CEST (02:00 never happens), 10:00 UTC in June = 12:00.")

## 2. Lag features without cheating
The day-ahead auction closes at **noon the day before**, so when forecasting hour *h* of tomorrow you know every price up to today 23:00, but not tomorrow's. That means lags of **24 h or more** only. Add `lag{L}` = the price *L* hours earlier for each `L` in `lags`, plus `mean_prev_day` = the mean of the 24 prices ending 24 hours earlier (i.e. yesterday-relative-to-target).

In [ ]:
def add_lags(df, lags=(24, 48, 72, 168), col="y"):
    # TODO 2: lag columns with .shift(L); mean_prev_day = col shifted 24 then rolling(24).mean()
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_lag():
    s = pd.DataFrame({"y": np.arange(200.0)})
    out = add_lags(s, lags=(24, 168))
    return out.lag24[30] == 6 and out.lag168[170] == 2 and np.isnan(out.lag168[100]) and out.mean_prev_day[47] == np.mean(np.arange(24))
check("lags", _t_lag, "lag24 at row 30 = value at row 6; mean_prev_day at row 47 = mean of rows 0..23.")

## 3. Baselines first
The metric is **MAE** in EUR/MWh (what a trader loses per MWh on average). Two baselines that any model must beat: *same hour yesterday* (`lag24`) and *same hour last week* (`lag168`).

In [ ]:
def mae(y_true, y_pred):
    # TODO 3: mean absolute error
    raise NotImplementedError  # TODO 3

In [ ]:
check("MAE", lambda: mae([1, 2, 3], [2, 2, 5]) == 1.0 and mae(pd.Series([0.0, -4.0]), np.array([0.0, 0.0])) == 2.0, "mean(|y − ŷ|): expect 1.0 and 2.0.")

SPLIT = pd.Timestamp("2025-01-01")
results = {}
if ready("calendar", "lags", "MAE"):
    feat = add_lags(add_calendar(df))
    train, test = feat[feat.index < SPLIT].dropna(), feat[feat.index >= SPLIT].dropna(subset=["y", "lag168"])
    results["naive: yesterday"] = mae(test.y, test.lag24)
    results["naive: last week"] = mae(test.y, test.lag168)
    for k, v in results.items(): print(f"{k:<22} MAE {v:6.2f} EUR/MWh")

## 4. Gradient boosting on price history
`HistGradientBoostingRegressor` handles nonlinearity and missing values out of the box (see [[Ensemble Methods]]).

In [ ]:
HISTORY = ["hour", "dow", "month", "weekend", "lag24", "lag48", "lag72", "lag168", "mean_prev_day"]
def fit_eval(features, name):
    m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0).fit(train[features], train.y)
    pred = m.predict(test[features]); results[name] = mae(test.y, pred); print(f"{name:<22} MAE {results[name]:6.2f} EUR/MWh")
    return m, pred

if ready("calendar", "lags", "MAE"):
    m_hist, pred_hist = fit_eval(HISTORY, "GBM: history")

No better than "same as yesterday" (in our run it's even a hair worse). History alone runs out of information. Prices in Denmark are driven by **how much wind is blowing**: lots of wind → cheap or even negative prices. Energinet publishes **day-ahead forecasts** of wind and solar production per price area (`Forecasts_Hour`), known *before* the auction, so they're fair game.

## 5. Add the weather forecast
`fc` is in long format (one row per hour × `ForecastType` ∈ {Offshore Wind, Onshore Wind, Solar}). Pivot `ForecastDayAhead` to wide, align it to `df.index` (`fc.t` is the UTC hour), and add `wind` = offshore + onshore (MWh) and `solar`.

In [ ]:
def add_weather(df, fc):
    # TODO 4: pivot_table(index="t", columns="ForecastType", values="ForecastDayAhead"), reindex to df.index; add wind and solar
    raise NotImplementedError  # TODO 4

In [ ]:
def _t_wx():
    t = pd.to_datetime(["2025-01-01 00:00", "2025-01-01 01:00"])
    toy = pd.DataFrame({"t": np.repeat(t, 3), "ForecastType": ["Offshore Wind", "Onshore Wind", "Solar"] * 2, "ForecastDayAhead": [100, 50, 0, 120, 60, 5.0]})
    out = add_weather(pd.DataFrame({"y": [1.0, 2.0]}, index=t), toy)
    return out.wind.tolist() == [150, 180] and out.solar.tolist() == [0, 5]
check("weather features", _t_wx, "wind = offshore + onshore → [150, 180]; solar → [0, 5].")

if ready("calendar", "lags", "MAE", "weather features"):
    feat = add_weather(add_lags(add_calendar(df)), fc)
    train, test = feat[feat.index < SPLIT].dropna(), feat[feat.index >= SPLIT].dropna(subset=["y", "lag168"])
    m_wx, pred_wx = fit_eval(HISTORY + ["wind", "solar"], "GBM: + wind & solar")
    plt.figure(figsize=(5, 3.5)); plt.scatter(train.wind, train.y, s=1, alpha=0.2); plt.xlabel("day-ahead wind forecast (MWh)"); plt.ylabel("price (EUR/MWh)")
    plt.ylim(-80, 400); plt.title("more wind → cheaper power"); plt.show()

In [ ]:
if ready("calendar", "lags", "MAE", "weather features"):
    check("weather beats history", lambda: results["GBM: + wind & solar"] < 0.85 * results["naive: yesterday"],
          "With wind & solar forecasts the model should cut the naive MAE by at least 15 %.")
    pd.Series(results).plot.barh(figsize=(6, 2.5), title="test MAE 2025 (EUR/MWh), lower is better"); plt.show()

## 6. Look at the errors
A single MAE hides a lot. Where does the model still fail?

In [ ]:
if ready("calendar", "lags", "MAE", "weather features"):
    err = test.assign(pred=pred_wx, abs_err=np.abs(test.y - pred_wx))
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
    err.groupby("hour").abs_err.mean().plot(ax=ax[0], marker="o", title="MAE by hour (Danish time)")
    worst = err.abs_err.resample("D").mean().idxmax()
    day = err.loc[str(worst.date())]; ax[1].plot(day.hour, day.y, "o-", label="actual"); ax[1].plot(day.hour, day.pred, "o-", label="forecast")
    ax[1].set_title(f"worst day: {worst.date()}"); ax[1].legend(); plt.show()
    print(f"hours with negative prices in 2025: {(test.y < 0).sum()}, of which predicted negative: {((test.y < 0) & (err.pred < 0)).sum()}")

## 7. Extension: does temperature help? (DMI open data)
The [DMI Open Data API](https://www.dmi.dk/frie-data) is free with no key. Below we fetch hourly mean temperature at **Flyvestation Aalborg** (station `06030`) and add it as a feature. Note that we use the *observed* temperature, which a real forecaster wouldn't have (they'd use a forecast), so this is an optimistic test.

In [ ]:
def dmi_temperature(station="06030", start="2023-10-01T00:00:00Z", end="2025-10-01T00:00:00Z"):
    f = CACHE / f"dmi_{station}_{start[:10]}_{end[:10]}.csv"
    if not f.exists():
        q = dict(stationId=station, parameterId="temp_mean_past1h", datetime=f"{start}/{end}", limit=300000)
        d = json.load(urllib.request.urlopen("https://opendataapi.dmi.dk/v2/metObs/collections/observation/items?" + urllib.parse.urlencode(q)))
        pd.DataFrame([x["properties"] for x in d["features"]])[["observed", "value"]].to_csv(f, index=False)
    t = pd.read_csv(f); s = t.assign(t=pd.to_datetime(t.observed).dt.tz_localize(None)).set_index("t").value.sort_index()
    return s[~s.index.duplicated()]

if ready("calendar", "lags", "MAE", "weather features"):
    temp = dmi_temperature().reindex(feat.index).interpolate(limit=3)
    feat["temp"] = temp
    train, test = feat[feat.index < SPLIT].dropna(), feat[feat.index >= SPLIT].dropna(subset=["y", "lag168"])
    fit_eval(HISTORY + ["wind", "solar", "temp"], "GBM: + temperature")
    print("→ Did it help? Think about why (hint: what does temperature change in a country with lots of wind and district heating?).")

## 8. Live: forecast tomorrow 🔴
Since **1 Oct 2025** prices are published in 15-minute resolution in the new `DayAheadPrices` dataset (the old hourly `Elspotprices` stopped). This cell fetches the last 3 weeks plus the wind/solar forecast for tomorrow, averages to hourly, and forecasts. Tomorrow's wind/solar forecasts appear in the dataset during the day: if the cell warns they're missing, try again later. Set `LIVE = True` to run it. Once the auction results for tomorrow are published (early afternoon), run it again and compare forecast vs actual!

In [ ]:
LIVE = False
if LIVE and ready("calendar", "lags", "MAE", "weather features"):
    today = pd.Timestamp.now(tz="Europe/Copenhagen").normalize()
    start, end = (today - pd.Timedelta(days=21)).strftime("%Y-%m-%d"), (today + pd.Timedelta(days=2)).strftime("%Y-%m-%d")
    q = lambda ds: energinet(ds, start, end)
    p15 = q("DayAheadPrices"); p15["t"] = pd.to_datetime(p15.TimeUTC).dt.floor("h")
    live = pd.DataFrame({"y": p15.groupby("t").DayAheadPriceEUR.mean()}).asfreq("h")
    tomorrow = pd.date_range((today + pd.Timedelta(days=1)).tz_convert("UTC").tz_localize(None), periods=24, freq="h")
    live = live.reindex(live.index.union(tomorrow))
    fcl = q("Forecasts_Hour"); fcl["t"] = pd.to_datetime(fcl.HourUTC)
    X = add_weather(add_lags(add_calendar(live)), fcl).loc[tomorrow]
    if X.wind.isna().any(): print("⚠️  Tomorrow's wind/solar forecast isn't in the dataset yet – try again later – the model falls back to history only.")
    X["pred"] = m_wx.predict(X[HISTORY + ["wind", "solar"]])
    print(X[["hour", "wind", "solar", "pred", "y"]].round(1).to_string())

<details><summary>▶ Solution</summary>

```python
def add_calendar(df):
    local = df.index.tz_localize("UTC").tz_convert("Europe/Copenhagen")
    out = df.copy()
    out["hour"], out["dow"], out["month"] = local.hour, local.dayofweek, local.month
    out["weekend"] = (local.dayofweek >= 5).astype(int)
    return out
```

```python
def add_lags(df, lags=(24, 48, 72, 168), col="y"):
    out = df.copy()
    for L in lags:
        out[f"lag{L}"] = out[col].shift(L)
    out["mean_prev_day"] = out[col].shift(24).rolling(24).mean()
    return out
```

```python
def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))
```

```python
def add_weather(df, fc):
    wide = fc.pivot_table(index="t", columns="ForecastType", values="ForecastDayAhead").reindex(df.index)
    out = df.copy()
    out["wind"] = wide[["Offshore Wind", "Onshore Wind"]].sum(axis=1, min_count=1)
    out["solar"] = wide["Solar"]
    return out
```
</details>

## Make it yours (stretch goals)
- **DK2 vs DK1:** set `PRICE_AREA = "DK2"`. Is Copenhagen's price easier or harder to forecast? Why? (Interconnectors to Sweden and Germany.)
- **Rolling-origin backtest:** retrain every month on all data before it, like a real deployment ([[Time Series Validation and Features]]).
- **Quantile forecasts:** `HistGradientBoostingRegressor(loss="quantile", quantile=0.1/0.9)` for an 80 % interval. When is it widest?
- **Your electricity bill:** shift a dishwasher run to the cheapest 3 hours of tomorrow. How much would you save per year?
- **Other Danish data:** Copenhagen's [fixed bike counts](https://www.opendata.dk/city-of-copenhagen/faste-cykeltaellinger) (Excel files, 2005–2014) + DMI rain. Do people cycle less when it rains? Energi Data Service also has CO₂ emission intensity (`CO2Emis`): forecast when it's *greenest* to charge an EV.